In [1]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# UPDATE THESE PATHS
# ============================================================

FILES = {
    "Chelannur": r"D:\NutriMap-AI\data\raw\Chelannur.csv",
    "Nedumangadu": r"D:\NutriMap-AI\data\raw\Nedumangadu.csv",
    "Pampady": r"D:\NutriMap-AI\data\raw\Pampady_clean.csv",
    "Vazhoor": r"D:\NutriMap-AI\data\raw\Vazhoor.csv"
}

# Kannur_Ezhome is intentionally excluded
# because it uses Available/Extractable nutrients
# which are not directly comparable.

OUTPUT_FILE = "Kerala_Soil_Clean_Final.csv"

In [2]:
# ============================================================
# CELL 2: LOAD DATASETS
# ============================================================

dfs = {}

for name, path in FILES.items():
    df = pd.read_csv(path)

    print(f"{name}")
    print("Shape:", df.shape)
    print("-"*50)

    dfs[name] = df

Chelannur
Shape: (1015, 21)
--------------------------------------------------
Nedumangadu
Shape: (1088, 21)
--------------------------------------------------
Pampady
Shape: (1926, 21)
--------------------------------------------------
Vazhoor
Shape: (1469, 21)
--------------------------------------------------


In [3]:
# ============================================================
# CELL 3: STANDARDIZE COLUMN NAMES
# ============================================================

COLUMN_MAPPING = {

    # Core Nutrients
    "pH": "pH",
    "ph": "pH",

    "EC": "EC",
    "ec": "EC",

    "Org__C": "Organic_Carbon",
    "Org _C": "Organic_Carbon",
    "OC_": "Organic_Carbon",
    "organic_carbon": "Organic_Carbon",

    "P": "P",
    "p": "P",

    "K": "K",
    "k": "K",

    "Ca": "Ca",
    "ca": "Ca",

    "Mg": "Mg",
    "mg": "Mg",

    "S": "S",
    "sulfur": "S",

    "Fe": "Fe",
    "iron": "Fe",

    "Mn": "Mn",
    "manganese": "Mn",

    "Zn": "Zn",
    "zinc": "Zn",

    "Cu": "Cu",
    "copper": "Cu",

    "B": "B",
    "boron": "B",

    # Location columns
    "District": "District",
    "district": "District",

    "Block": "Block",
    "block": "Block",

    "Latitude": "Latitude",
    "latitude": "Latitude",

    "Longitude": "Longitude",
    "longitude": "Longitude"
}

for key in dfs:

    df = dfs[key]

    df.rename(columns=COLUMN_MAPPING, inplace=True)

    dfs[key] = df

In [4]:
# ============================================================
# CELL 4: KEEP ONLY CONSISTENT FEATURES
# ============================================================

FINAL_COLUMNS = [

    # Location
    "District",
    "Block",
    "Latitude",
    "Longitude",

    # Soil Features
    "pH",
    "EC",
    "Organic_Carbon",
    "P",
    "K",
    "Ca",
    "Mg",
    "S",
    "Fe",
    "Mn",
    "Zn",
    "Cu",
    "B"
]

clean_dfs = {}

for key, df in dfs.items():

    available_cols = [c for c in FINAL_COLUMNS if c in df.columns]

    temp = df[available_cols].copy()

    clean_dfs[key] = temp

    print(key, temp.shape)

Chelannur (1015, 17)
Nedumangadu (1088, 17)
Pampady (1926, 17)
Vazhoor (1469, 17)


In [5]:
# ============================================================
# CELL 5: CONVERT NUTRIENTS TO NUMERIC
# ============================================================

nutrient_cols = [
    "pH",
    "EC",
    "Organic_Carbon",
    "P",
    "K",
    "Ca",
    "Mg",
    "S",
    "Fe",
    "Mn",
    "Zn",
    "Cu",
    "B"
]

for key in clean_dfs:

    df = clean_dfs[key]

    for col in nutrient_cols:

        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    clean_dfs[key] = df

In [6]:
# ============================================================
# CELL 6: MERGE DATASETS
# ============================================================

final_df = pd.concat(
    clean_dfs.values(),
    ignore_index=True
)

print("Merged Shape:", final_df.shape)

Merged Shape: (5498, 17)


In [7]:
# ============================================================
# CELL 7: CLEAN LATITUDE/LONGITUDE
# ============================================================

final_df["Latitude"] = pd.to_numeric(
    final_df["Latitude"],
    errors="coerce"
)

final_df["Longitude"] = pd.to_numeric(
    final_df["Longitude"],
    errors="coerce"
)

# Remove invalid coordinates

final_df = final_df[
    final_df["Latitude"].between(-90, 90)
]

final_df = final_df[
    final_df["Longitude"].between(-180, 180)
]

print("After coordinate cleaning:")
print(final_df.shape)

After coordinate cleaning:
(5498, 17)


In [8]:
# ============================================================
# CELL 8: REMOVE DUPLICATES
# ============================================================

before = len(final_df)

final_df = final_df.drop_duplicates()

after = len(final_df)

print("Duplicates Removed:", before - after)
print("Remaining Rows:", after)

Duplicates Removed: 0
Remaining Rows: 5498


In [9]:
# ============================================================
# CELL 9: REMOVE INCOMPLETE RECORDS
# ============================================================

critical_columns = [
    "District",
    "Block",
    "Latitude",
    "Longitude"
]

final_df = final_df.dropna(
    subset=critical_columns
)

print("After NA removal:")
print(final_df.shape)

After NA removal:
(5498, 17)


In [10]:
# ============================================================
# CELL 10: STANDARDIZE TEXT COLUMNS
# ============================================================

final_df["District"] = (
    final_df["District"]
    .astype(str)
    .str.strip()
    .str.title()
)

final_df["Block"] = (
    final_df["Block"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [11]:
# ============================================================
# FIX DISTRICT AND BLOCK DATA QUALITY ISSUES
# ============================================================

print("Before Cleaning")
print("\nDistrict:")
print(final_df["District"].value_counts(dropna=False))

print("\nBlock:")
print(final_df["Block"].value_counts(dropna=False))


# ------------------------------------------------------------
# 1. Fix common spelling mistakes
# ------------------------------------------------------------

block_corrections = {
    "Pamapady": "Pampady"
}

district_corrections = {
    "Trivandrum": "Thiruvananthapuram"  # optional standardization
}

final_df["Block"] = final_df["Block"].replace(block_corrections)
final_df["District"] = final_df["District"].replace(district_corrections)


# ------------------------------------------------------------
# 2. Remove obvious garbage District values
# ------------------------------------------------------------
# These are clearly corrupted records and should not be used
# in ML/DL training.

bad_districts = [
    "95",
    "189",
    "Padd",
    "Kottayam+A330:Ac345A330:Ac345"
]

final_df = final_df[
    ~final_df["District"].astype(str).str.strip().isin(bad_districts)
]


# ------------------------------------------------------------
# 3. Keep only valid Districts
# ------------------------------------------------------------
# Safer than manually removing bad values because it catches
# future corruption too.

valid_districts = [
    "Kottayam",
    "Kozhikode",
    "Trivandrum",            # remove if using Thiruvananthapuram
    "Thiruvananthapuram"     # remove if keeping Trivandrum
]

final_df = final_df[
    final_df["District"].isin(valid_districts)
]


# ------------------------------------------------------------
# 4. Keep only valid Blocks
# ------------------------------------------------------------

valid_blocks = [
    "Pampady",
    "Vazhoor",
    "Nedumangad",
    "Chelannur"
]

final_df = final_df[
    final_df["Block"].isin(valid_blocks)
]


# ------------------------------------------------------------
# 5. Reset index after filtering
# ------------------------------------------------------------

final_df = final_df.reset_index(drop=True)


# ------------------------------------------------------------
# 6. Verify results
# ------------------------------------------------------------

print("\n\nAfter Cleaning")
print("\nDistrict:")
print(final_df["District"].value_counts())

print("\nBlock:")
print(final_df["Block"].value_counts())

print("\nFinal Shape:", final_df.shape)

Before Cleaning

District:
District
Kottayam                         3393
Trivandrum                       1088
Kozhikode                        1013
189                                 1
95                                  1
Padd                                1
Kottayam+A330:Ac345A330:Ac345       1
Name: count, dtype: int64

Block:
Block
Pampady       1814
Vazhoor       1469
Nedumangad    1088
Chelannur     1015
Pamapady       112
Name: count, dtype: int64


After Cleaning

District:
District
Kottayam              3393
Thiruvananthapuram    1088
Kozhikode             1013
Name: count, dtype: int64

Block:
Block
Pampady       1925
Vazhoor       1468
Nedumangad    1088
Chelannur     1013
Name: count, dtype: int64

Final Shape: (5494, 17)


In [12]:
# ============================================================
# CELL 11: QUALITY CHECKS
# ============================================================

print("\nFinal Shape")
print(final_df.shape)

print("\nColumns")
print(final_df.columns.tolist())

print("\nMissing Values")
print(final_df.isnull().sum())

print("\nDuplicate Rows")
print(final_df.duplicated().sum())


Final Shape
(5494, 17)

Columns
['District', 'Block', 'Latitude', 'Longitude', 'pH', 'EC', 'Organic_Carbon', 'P', 'K', 'Ca', 'Mg', 'S', 'Fe', 'Mn', 'Zn', 'Cu', 'B']

Missing Values
District          0
Block             0
Latitude          0
Longitude         0
pH                0
EC                0
Organic_Carbon    0
P                 0
K                 0
Ca                1
Mg                0
S                 0
Fe                0
Mn                0
Zn                0
Cu                0
B                 0
dtype: int64

Duplicate Rows
0


In [13]:
# ============================================================
# CELL 12: EXPORT CLEAN CSV
# ============================================================

final_df = final_df.sort_values(
    ["District", "Block"]
).reset_index(drop=True)

final_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("="*60)
print("SUCCESS")
print("="*60)
print("Saved:", OUTPUT_FILE)
print("Rows :", final_df.shape[0])
print("Cols :", final_df.shape[1])

final_df.head()

SUCCESS
Saved: Kerala_Soil_Clean_Final.csv
Rows : 5494
Cols : 17


,District,Block,Latitude,Longitude,pH,EC,Organic_Carbon,P,K,Ca,Mg,S,Fe,Mn,Zn,Cu,B
0,Kottayam,Pampady,9.65674,76.63473,4.5,0.22,3.40,183.97,372.96,313.90,205.45,13.88,5.960,7.724,0.210,0.20325,0.049
1,Kottayam,Pampady,9.65622,76.63101,5.6,0.25,1.74,57.31,268.24,1000.00,200.65,17.90,3.560,7.662,0.263,0.11675,0.047
2,Kottayam,Pampady,9.65564,76.63068,5.9,0.24,1.31,79.07,565.26,949.95,254.48,19.73,3.200,15.560,0.300,0.22250,0.138
3,Kottayam,Pampady,9.65670,76.63118,6.1,0.31,1.86,68.33,440.72,1000.00,404.48,30.98,5.236,7.924,0.982,0.10800,0.080
4,Kottayam,Pampady,9.65708,76.63483,5.8,0.26,0.95,153.46,227.70,614.20,257.68,24.32,2.952,8.626,0.601,0.19775,0.041
